# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [ ]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests
BASE=Path('/content/flasheats_classroom')
if not BASE.exists():
    from google.colab import files
    print('Upload FlashEats_Classroom_Pack.zip')
    uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith('.zip'))
    with zipfile.ZipFile(zip_name) as z: z.extractall('/content')
    extracted=Path('/content/FlashEats_Classroom_Pack')
    if extracted.exists(): extracted.rename(BASE)
print(BASE, BASE.exists())

## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

In [ ]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

In [ ]:
query='''
-- YOUR SQL HERE
SELECT * FROM orders LIMIT 5;
'''
pd.read_sql(query,con)

### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [ ]:
# Your analysis here

## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [ ]:
tickets=pd.read_csv(BASE/'data'/'support_tickets.csv'); tickets.head()

## Challenge 4 — Retrieve Dispatch data reliably

In [ ]:
!pip -q install flask
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

In [ ]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [ ]:
RAW_DIR=BASE/'student_output'/'raw_dispatch'; RAW_DIR.mkdir(parents=True,exist_ok=True)
def fetch_all_dispatch_orders():
    records=[]
    # TODO
    return records

## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [ ]:
with open(BASE/'data'/'driver_events.json') as f: driver_events=json.load(f)
driver_events[0]

## Final FDE recommendation
Prepare one slide: problem size, evidence, uncertainty, missing instrumentation, next step.

**Would you build the AI delay predictor now? Why or why not?**